# TP final · Opción B · Efecto de la música

Problema: predecir `Music effects` (`Improve`, `No effect`, `Worsen`) a partir de los hábitos de escucha. Es clasificación. La métrica que va a decidir es el F1 macro de test.

Esta parte solo mira los datos. No imputa, no descarta filas y no entrena.

Fuente: [Music & Mental Health Survey Results](https://www.kaggle.com/datasets/catherinerasgaitis/mxmh-survey-results), Catherine Rasgaitis, licencia CC0. El CSV está en `data/mxmh_survey_results.csv`.


## Widgets

- **Archivo CSV.** Qué archivo lee Spark. Ya viene con la ruta del CSV de este repo.
- **Puntajes que entran.** Si `Anxiety`, `Depression`, `Insomnia` y `OCD` se usan como variables. `ninguno` no usa ninguno. `los cuatro` usa todos. `seleccionados` usa solo los que escribas en **Puntajes seleccionados**.
- **Puntajes seleccionados.** Nombres separados por coma, por ejemplo `Depression, Anxiety`. Solo se lee si arriba elegiste `seleccionados`.

Esta sección todavía no entrena: imprime la lista para comprobarla.


In [ ]:
def ruta_csv_default():
    notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    carpeta = "/Workspace" + notebook.rsplit("/", 1)[0]
    return "file:" + carpeta + "/data/mxmh_survey_results.csv"

for nombre in ("csv_path", "puntajes_salud", "puntajes_algunas"):
    try:
        dbutils.widgets.remove(nombre)
    except Exception:
        pass

dbutils.widgets.text("csv_path", ruta_csv_default(), "Ruta CSV")
dbutils.widgets.dropdown("puntajes_salud", "ninguno", ["ninguno", "seleccionados", "los cuatro"], "Puntajes que entran")
dbutils.widgets.text("puntajes_algunas", "Depression", "Puntajes seleccionados")


In [ ]:
from pyspark.sql import functions as F

SALUD = ["Anxiety", "Depression", "Insomnia", "OCD"]

def resolver_csv():
    pedido = dbutils.widgets.get("csv_path").strip()
    return pedido or ruta_csv_default()

csv_path = resolver_csv()
print("csv:", csv_path)

crudo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)
print(f"filas: {crudo.count():,}  columnas: {len(crudo.columns)}")


In [ ]:
def columnas_salud():
    modo = dbutils.widgets.get("puntajes_salud")
    if modo == "ninguno":
        return []
    if modo == "los cuatro":
        return list(SALUD)
    pedidas = [parte.strip() for parte in dbutils.widgets.get("puntajes_algunas").split(",") if parte.strip()]
    desconocidas = [nombre for nombre in pedidas if nombre not in SALUD]
    if desconocidas:
        raise ValueError(f"No son puntajes de salud: {desconocidas}. Opciones: {SALUD}")
    return pedidas

elegidas = columnas_salud()
print("modo:", dbutils.widgets.get("puntajes_salud"))
print("puntajes que entran:", elegidas if elegidas else "(ninguno)")


## 1. Qué hay en la tabla

Conteos, nulos y las tres clases. Una celda vacía cuenta como faltante, igual que un nulo. En cada columna se listan hasta 8 valores, ordenados por frecuencia. No se rellena nada.


In [ ]:
display(crudo.limit(5))

TOPE = 8
filas = []
for nombre in crudo.columns:
    texto = F.trim(F.col(nombre).cast("string"))
    vacio = F.col(nombre).isNull() | (texto == "")
    presentes = crudo.where(~vacio)
    conteos = (
        presentes.groupBy(texto.alias("valor"))
        .count()
        .orderBy(F.desc("count"), "valor")
        .collect()
    )
    mostrados = conteos[:TOPE]
    partes = [f"{fila.valor} ({fila['count']})" for fila in mostrados]
    resto = len(conteos) - len(mostrados)
    if resto > 0:
        partes.append(f"y {resto} más")
    filas.append((nombre, int(crudo.where(vacio).count()), len(conteos), ", ".join(partes)))

display(spark.createDataFrame(filas, ["columna", "nulos", "distintos", "valores"]))


In [ ]:
etiqueta = F.when(
    F.col("Music effects").isNull() | (F.trim(F.col("Music effects")) == ""),
    "(vacio)",
).otherwise(F.col("Music effects"))
display(
    crudo.groupBy(etiqueta.alias("Music effects"))
    .count()
    .orderBy(F.desc("count"))
)


## 2. Age, horas y BPM

Resumen numérico y las filas de BPM vacías, no numéricas o fuera de 40–250. Ese rango no viene de la encuesta ni de la consigna: un tema suele ir de 60 a 180, y 40–250 es un margen ancho para marcar valores que no son un tempo, como 0 o 999999999. No decide el arreglo. Esta celda no modifica la tabla.


In [ ]:
def resumen_numerico(nombre):
    texto = F.trim(F.col(nombre).cast("string"))
    numero = F.col(nombre).cast("double")
    vacio = F.col(nombre).isNull() | (texto == "")
    no_numerico = ~vacio & numero.isNull()
    print(nombre)
    display(crudo.select(
        F.count("*").alias("filas"),
        F.count(F.when(vacio, 1)).alias("vacios"),
        F.count(F.when(no_numerico, 1)).alias("no_numerico"),
        F.min(numero).alias("minimo"),
        F.max(numero).alias("maximo"),
    ))

for nombre in ("Age", "Hours per day", "BPM"):
    resumen_numerico(nombre)

bpm = F.col("BPM").cast("double")


In [ ]:
raros = crudo.where(
    F.col("BPM").isNull()
    | (F.trim(F.col("BPM").cast("string")) == "")
    | bpm.isNull()
    | (bpm < 40)
    | (bpm > 250)
)
print(f"filas con BPM vacio, no numerico o fuera de 40-250: {raros.count()}")
display(raros.select("Age", "Fav genre", "BPM", "Music effects").orderBy(bpm.asc_nulls_first()))


## Modelos previstos

Cuando el arreglo esté decidido, los cuatro ven la misma matriz y el mismo corte 80/20 estratificado (`random_state=7`). Los tres de árboles usan el balanceo de clases. Gana el mayor F1 macro de test.

| Run | Modelo |
|---|---|
| `logistica` | Regresión logística, `C=1` |
| `random_forest` | Random forest |
| `lightgbm` | LightGBM |
| `catboost` | CatBoost |

El entrenamiento queda para después de mirar esta exploración. No se imputa en este notebook todavía.
